# Satellite Orbit Explorer · built during Space Career Week 2026 · GNSS Academy

Eight stages:
1. Imports and configuration
2. Load GP JSON
3. Parse orbital elements
4. Time window
5. Propagate
6. Orbit summary
7. Plots
8. KML and report

**Model and limitations:** Educational two-body Kepler orbit, spherical Earth and simple Greenwich rotation. GP records provide SGP4 mean elements, so predictions using this simplified model are approximations, **not precision orbit predictions**.

All helper functions will be defined inside notebook cells (no external `.py` modules).

In [ ]:
import json
import math
import time
import zipfile
import xml.etree.ElementTree as ET
from datetime import datetime, timedelta, timezone
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
import requests

# Educational two-body orbital model constants (km, s).
MU_EARTH_KM3_S2 = 398600.4418
EARTH_RADIUS_KM = 6378.137

# Colab input/output directories.
BASE_DIR = Path("/content/satellite_orbits")
IN_DIR = BASE_DIR / "in"
OUT_DIR = BASE_DIR / "out"
IN_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Stage 1 configuration. LIVE_GROUP is reserved for an optional later extension.
STEP_SECONDS = 5
DURATION_HOURS = 24
LIVE_GROUP = None

print("Satellite Orbit Explorer — configuration")
print(f"Input directory:  {IN_DIR}")
print(f"Output directory: {OUT_DIR}")
print(f"Time window:      {DURATION_HOURS} hours")
print(f"Sampling step:    {STEP_SECONDS} seconds")
print(f"Live group:       {LIVE_GROUP}")
print("Conventions: UTC-aware datetimes, km, km/s")


## Stage 2 — Load GP JSON

OMM JSON uses named fields rather than fixed-width TLE lines, making each orbital element easier to identify. It also supports ISO dates and six-digit catalog numbers without the old TLE format's limitations.

In [ ]:
# Use the three original workshop inputs; do not synthesize data.
INPUT_NAMES = ("orbital_elements.json", "data_manifest.json", "world_map.txt")

def extract_inputs():
    if all((IN_DIR / name).is_file() for name in INPUT_NAMES):
        return

    print("Drag orbit_workshop_inputs.zip into /content using the Colab Files panel.")
    deadline = time.monotonic() + 15 * 60
    archive = None
    while time.monotonic() < deadline:
        for candidate in sorted(Path("/content").glob("orbit_workshop_inputs*.zip")):
            if zipfile.is_zipfile(candidate):
                archive = candidate
                break
        if archive is not None:
            break
        time.sleep(2)
    if archive is None:
        raise TimeoutError("No completed orbit_workshop_inputs*.zip appeared in /content within 15 minutes.")

    with zipfile.ZipFile(archive) as zf:
        # Accept files at archive root or in a folder, without extracting other entries.
        entries = {}
        for info in zf.infolist():
            if not info.is_dir() and Path(info.filename).name in INPUT_NAMES:
                name = Path(info.filename).name
                if name in entries:
                    raise ValueError(f"Duplicate input in ZIP: {name}")
                entries[name] = info
        missing = set(INPUT_NAMES) - entries.keys()
        if missing:
            raise ValueError(f"Missing files in ZIP: {', '.join(sorted(missing))}")
        for name in INPUT_NAMES:
            # Read fully before writing: catches truncated archives and CRC errors.
            content = zf.read(entries[name])
            (IN_DIR / name).write_bytes(content)


def read_gp_json(path):
    with open(path, encoding="utf-8") as stream:
        records = json.load(stream)
    if not isinstance(records, list) or not records:
        raise ValueError("GP JSON must contain a non-empty list of records.")
    required = {
        "OBJECT_NAME", "NORAD_CAT_ID", "EPOCH", "MEAN_MOTION",
        "ECCENTRICITY", "INCLINATION", "RA_OF_ASC_NODE",
        "ARG_OF_PERICENTER", "MEAN_ANOMALY",
    }
    for index, record in enumerate(records, start=1):
        if not isinstance(record, dict):
            raise ValueError(f"GP record {index} is not a JSON object.")
        missing = required - record.keys()
        if missing:
            raise ValueError(f"GP record {index} missing fields: {', '.join(sorted(missing))}")
    return records


extract_inputs()
sats = read_gp_json(IN_DIR / "orbital_elements.json")
with open(IN_DIR / "data_manifest.json", encoding="utf-8") as stream:
    data_manifest = json.load(stream)
if not isinstance(data_manifest, dict) or not data_manifest.get("retrieved_utc"):
    raise ValueError("data_manifest.json must contain retrieved_utc.")
source_note = f"CelesTrak GP JSON (OMM fields), retrieved UTC: {data_manifest['retrieved_utc']}"
print(source_note)
for sat in sats:
    print(f"{sat['OBJECT_NAME']} | NORAD {sat['NORAD_CAT_ID']} | EPOCH {sat['EPOCH']}")
